In [ ]:

# ============================================================
# SUB-TASK 1: LOAD DATA & INITIALIZE THE EVENT IMPACT MODEL
# ============================================================
# This is always the first step - we need to load the real data
# and create our model object before doing anything else.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("SUB-TASK 1: Loading Data & Initializing Model")
print("=" * 70)
print()

# Step 1.1: Load the main data sheet
print("Step 1.1: Loading main data sheet...")
df = pd.read_excel('/mnt/agents/upload/ethiopia_fi_unified_data.xlsx')
df = df.iloc[1:].reset_index(drop=True)  # Remove header row
df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
print(f"  ✓ Loaded {len(df)} records")
print(f"  ✓ Record types: {df['record_type'].value_counts().to_dict()}")
print()

# Step 1.2: Load the impact links sheet
print("Step 1.2: Loading impact links sheet...")
df_impact = pd.read_excel('/mnt/agents/upload/ethiopia_fi_unified_data.xlsx', sheet_name=1)
df_impact = df_impact.iloc[1:].reset_index(drop=True)
print(f"  ✓ Loaded {len(df_impact)} impact links")
print()

# Step 1.3: Load reference codes
print("Step 1.3: Loading reference codes...")
df_ref = pd.read_excel('/mnt/agents/upload/reference_codes.xlsx')
df_ref = df_ref.iloc[1:].reset_index(drop=True)
print(f"  ✓ Loaded {len(df_ref)} reference codes")
print()

# Step 1.4: Separate record types for easy access
observations = df[df['record_type'] == 'observation'].copy()
events = df[df['record_type'] == 'event'].copy()
targets = df[df['record_type'] == 'target'].copy()

print("Step 1.4: Data separated by record type:")
print(f"  • Observations: {len(observations)}")
print(f"  • Events: {len(events)}")
print(f"  • Targets: {len(targets)}")
print()

# Step 1.5: Initialize the EventImpactModel
print("Step 1.5: Initializing EventImpactModel...")

class EventImpactModel:
    """Models how events affect financial inclusion indicators over time."""
    
    def __init__(self, df_data, df_impact, ramp_up_months=3, decay_halflife=12):
        self.df = df_data.copy()
        self.df_impact = df_impact.copy()
        self.ramp_up = ramp_up_months
        self.decay_halflife = decay_halflife
        
        self.df['observation_date'] = pd.to_datetime(self.df['observation_date'], errors='coerce')
        self.observations = self.df[self.df['record_type'] == 'observation'].copy()
        self.events = self.df[self.df['record_type'] == 'event'].copy()
        self.targets = self.df[self.df['record_type'] == 'target'].copy()
        
        self._build_event_indicator_map()
        print(f"  ✓ Model initialized (ramp={ramp_up_months}mo, decay_halflife={decay_halflife}mo)")
    
    def _build_event_indicator_map(self):
        """Build mapping of events to their affected indicators."""
        self.event_indicator_map = {}
        for _, link in self.df_impact.iterrows():
            parent_id = link['parent_id']
            if parent_id not in self.event_indicator_map:
                self.event_indicator_map[parent_id] = []
            self.event_indicator_map[parent_id].append({
                'indicator': link['related_indicator'],
                'impact_direction': link['impact_direction'],
                'impact_estimate': link['impact_estimate'],
                'lag_months': link['lag_months'],
                'confidence': link['confidence'],
                'evidence_basis': link['evidence_basis']
            })
    
    def event_effect(self, event_date, lag_months, impact_estimate, current_date):
        """Calculate active effect of an event at a given date."""
        if pd.isna(event_date) or pd.isna(current_date):
            return 0.0
        
        months_since_event = (current_date - event_date).days / 30.44
        
        # Phase 1: Before lag - no effect
        if months_since_event < lag_months:
            return 0.0
        
        months_since_effect_start = months_since_event - lag_months
        
        # Phase 2: Ramp up
        if months_since_effect_start < self.ramp_up:
            ramp_progress = months_since_effect_start / self.ramp_up
            return impact_estimate * ramp_progress
        
        # Phase 3: Peak and decay
        months_at_peak = months_since_effect_start - self.ramp_up
        decay_factor = 0.5 ** (months_at_peak / self.decay_halflife)
        return impact_estimate * decay_factor

# Create the model instance
model = EventImpactModel(df, df_impact)

print()
print("=" * 70)
print("✅ SUB-TASK 1 COMPLETE: Data loaded & model ready")
print("=" * 70)
